# Aula 06: tendências centrais

Este notebook acompanha a aula sobre tendências centrais. Vamos usar uma base didática de faixas de streaming para estudar duração de músicas, média, mediana, quartis, dispersão e efeitos de valores extremos.

## Pergunta motivadora

Como podemos resumir a duração das faixas de uma coleção sem esconder informação importante sobre a distribuição?

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

rng = np.random.default_rng(2026)
n = 600

generos = rng.choice(
    ["pop", "rap", "rock", "eletronica", "sertanejo"],
    size=n,
    p=[0.32, 0.22, 0.18, 0.16, 0.12]
)

duracao = rng.normal(loc=3.6, scale=0.75, size=n).clip(0.8, None)
duracao += np.where(generos == "eletronica", rng.normal(0.7, 0.35, n), 0)
duracao += np.where(generos == "rap", rng.normal(-0.25, 0.20, n), 0)

# Algumas faixas muito longas criam uma cauda à direita.
idx_longas = rng.choice(np.arange(n), size=10, replace=False)
duracao[idx_longas] += rng.uniform(3.0, 7.0, size=len(idx_longas))

faixas = pd.DataFrame({
    "faixa": [f"faixa_{i:03d}" for i in range(1, n + 1)],
    "genero": generos,
    "duracao_min": duracao,
    "popularidade": rng.integers(1, 101, size=n),
})

faixas.head()

## Amostra e população

A tabela acima é uma **amostra**. A população poderia ser "todas as faixas disponíveis em uma plataforma" ou "todas as faixas escutadas por estudantes da turma". A interpretação depende dessa definição.

In [ ]:
faixas["duracao_min"].describe().round(2)

## Histograma

Antes de calcular uma média, visualize a distribuição. O histograma mostra onde os valores se concentram e se há caudas ou valores extremos.

In [ ]:
ax = faixas["duracao_min"].plot.hist(bins=30, color="#0f6b78", edgecolor="white")
ax.set_title("Duração das faixas")
ax.set_xlabel("duração (minutos)")
ax.set_ylabel("número de faixas")

## CDF empírica

A CDF empírica mostra a fração dos dados menor ou igual a cada valor. Ela é útil para leituras como: "qual fração das faixas tem até 4 minutos?"

In [ ]:
x = np.sort(faixas["duracao_min"])
y = np.arange(1, len(x) + 1) / len(x)

ax = pd.Series(y, index=x).plot(color="#d95f02")
ax.set_title("CDF empírica da duração")
ax.set_xlabel("duração (minutos)")
ax.set_ylabel("fração acumulada")

## Média e mediana

A média usa todos os valores. A mediana é o ponto que divide os dados ordenados em duas metades. Quando há valores extremos, elas podem contar histórias diferentes.

In [ ]:
media = faixas["duracao_min"].mean()
mediana = faixas["duracao_min"].median()

pd.Series({"média": media, "mediana": mediana}).round(2)

In [ ]:
ax = faixas["duracao_min"].plot.hist(bins=30, color="#dbe3e7", edgecolor="white")
ax.axvline(media, color="#b23b3b", linewidth=3, label=f"média = {media:.2f}")
ax.axvline(mediana, color="#0f6b78", linewidth=3, label=f"mediana = {mediana:.2f}")
ax.set_title("Média e mediana na distribuição")
ax.set_xlabel("duração (minutos)")
ax.legend()

## Um valor extremo

Vamos adicionar uma faixa sintética de 45 minutos. Observe o que muda na média e na mediana.

In [ ]:
com_outlier = pd.concat([
    faixas,
    pd.DataFrame({
        "faixa": ["faixa_gigante"],
        "genero": ["experimental"],
        "duracao_min": [45.0],
        "popularidade": [12],
    })
], ignore_index=True)

pd.DataFrame({
    "sem_outlier": [faixas["duracao_min"].mean(), faixas["duracao_min"].median()],
    "com_outlier": [com_outlier["duracao_min"].mean(), com_outlier["duracao_min"].median()],
}, index=["média", "mediana"]).round(2)

## Quartis e boxplot

Quartis resumem a distribuição em pontos de corte: 25%, 50% e 75%. O boxplot usa esses pontos para mostrar centro, dispersão e possíveis valores extremos.

In [ ]:
faixas["duracao_min"].quantile([0.25, 0.50, 0.75]).round(2)

In [ ]:
ax = sns.boxplot(data=faixas, x="genero", y="duracao_min", color="#dbe3e7")
ax.set_title("Duração por gênero")
ax.set_xlabel("")
ax.set_ylabel("duração (minutos)")

## Dispersão

Tendência central não basta. Duas distribuições podem ter a mesma média e espalhamentos muito diferentes.

In [ ]:
pd.Series({
    "mínimo": faixas["duracao_min"].min(),
    "máximo": faixas["duracao_min"].max(),
    "intervalo": faixas["duracao_min"].max() - faixas["duracao_min"].min(),
    "variância": faixas["duracao_min"].var(),
    "desvio padrão": faixas["duracao_min"].std(),
}).round(2)

## Transformação logarítmica

Quando uma variável tem muitos valores pequenos e poucos valores enormes, uma escala logarítmica pode revelar estrutura que fica escondida na escala original.

In [ ]:
palavras = pd.DataFrame({
    "palavra": [f"palavra_{i}" for i in range(1, 501)],
    "num_faixas": np.maximum(1, (600 / np.arange(1, 501) ** 0.9).astype(int))
})

ax = palavras.plot.scatter(x="palavra", y="num_faixas", figsize=(8, 3), color="#0f6b78")
ax.set_xticks([])
ax.set_title("Ocorrência de palavras em faixas")
ax.set_xlabel("palavras ordenadas")
ax.set_ylabel("número de faixas")

In [ ]:
palavras["rank"] = np.arange(1, len(palavras) + 1)
ax = palavras.plot.scatter(x="rank", y="num_faixas", logx=True, logy=True, color="#d95f02")
ax.set_title("Mesmos dados em escala log-log")
ax.set_xlabel("rank da palavra")
ax.set_ylabel("número de faixas")

## Outras médias

- Média aritmética: quando somar valores na mesma unidade faz sentido.
- Média geométrica: útil para combinar escalas normalizadas ou crescimento multiplicativo.
- Média harmônica: útil para taxas, como velocidades médias em trajetos de mesma distância.

## Para praticar

1. Escolha outra coluna numérica e faça histograma e CDF.
2. Compare média e mediana por gênero.
3. Adicione outro valor extremo e explique o efeito.
4. Escreva uma recomendação: qual resumo você usaria para comunicar a duração típica das faixas?